# 01 - Feature Engineering
### DI Leather · Forecasting weekly production output (`actual_qty`)

**Purpose:** turn the cleaned dataset into the final, model-ready feature table.

| | |
|---|---|
| **Input** | `DI_Leather_preprocessed.csv` (cleaned weekly production records) |
| **Output** | `final_features.csv` |
| **Target** | `actual_qty` — the number of shoes actually produced |
| **Next notebook** | `02_Model_Selection_and_Training.ipynb` |

**Steps in this notebook**
1. Load the data
2. Parse dates and sort chronologically
3. Create three new capacity/intensity features
4. Check the new features
5. Remove columns that must not be model inputs
6. Review the final feature set
7. Run quality checks and save

> **What this notebook does *not* do:** scaling, one-hot encoding and target encoding are deliberately left to the modelling pipeline in Notebook 2. Those transformations learn from the data (means, categories, target averages), so they must be fitted on the *training* weeks only. Doing them here, before the train/test split, would leak information from the test weeks.


---
## Step 1 - Load the data


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("DI_Leather_preprocessed.csv")

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head()

Rows: 9495 | Columns: 22


,financial_year,financial_month,week_no,week_of_month,week_start_date,week_end_date,factory_unit,product_family,item_name,size_id,...,rate,actual_qty,no_of_workers,working_hours,total_duration_per_shoe_mins,machine_capacity_per_day,machine_count,production_gap_qty,bottleneck_flag,order_category
0,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,12,...,4363.35,38,15,647.82,114.50,200,5,-3,True,Others
1,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - BLACK,7,...,4870.92,33,23,844.78,123.14,200,5,8,True,Others
2,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - GREEN,9,...,7003.71,44,15,673.07,89.04,200,3,-29,False,Others
3,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - BLACK,13,...,4676.91,31,11,550.89,125.18,200,3,-26,False,Forces/Police
4,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - L - BLACK,13,...,3803.13,36,18,813.02,117.33,200,5,-25,True,Forces/Police


The raw table has one row per **week × factory unit × item × size**. Each row already contains the planned and actual quantity, workforce, machine and process-time information.


---
## Step 2 - Parse dates and sort chronologically
The model will later be tested on the **most recent** weeks, so rows must be in time order. `week_start_date` is converted to a real date and used to sort the table. It is kept in the saved file only as an *ordering key* - it is never used as a model input.


In [2]:
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df = df.sort_values('week_start_date').reset_index(drop=True)

print("First week:", df['week_start_date'].min().date())
print("Last week :", df['week_start_date'].max().date())
print("Sorted chronologically:", df['week_start_date'].is_monotonic_increasing)

First week: 2022-04-18
Last week : 2026-03-30
Sorted chronologically: True


---
## Step 3 - Create new features
Three ratios are built from the workforce, machine and process-time columns. They describe the factory's **capacity and resource intensity**, which a model can use more easily than the raw columns on their own.

> **Assumption to confirm with the data provider:** `working_hours` and `total_duration_per_shoe_mins` are treated as *planned / standard* figures that are known **before** production starts. If either is actually recorded after the week ends, these three features (and the model) would leak the outcome.


### 3.1 `theoretical_max_production`
`working_hours × 60 ÷ total_duration_per_shoe_mins`

Working hours are converted to minutes and divided by the standard minutes needed per shoe. The result is how many shoes the available labour time could produce if all of it went into this shoe type - a **capacity indicator**.


In [3]:
df['theoretical_max_production'] = (df['working_hours'] * 60) / df['total_duration_per_shoe_mins']

### 3.2 `machine_worker_ratio`
`machine_count ÷ no_of_workers`

Machines available per worker - a measure of **equipment intensity**.


In [4]:
df['machine_worker_ratio'] = df['machine_count'] / df['no_of_workers']

### 3.3 `avg_hours_per_worker`
`working_hours ÷ no_of_workers`

Average hours each worker contributes - a measure of **labour intensity**.


In [5]:
df['avg_hours_per_worker'] = df['working_hours'] / df['no_of_workers']

---
## Step 4 - Check the new features
Confirm the three features contain no missing or infinite values (a zero in a denominator would create them) and look at their ranges.


In [6]:
engineered = ['theoretical_max_production', 'machine_worker_ratio', 'avg_hours_per_worker']

print("Missing values :", int(df[engineered].isna().sum().sum()))
print("Infinite values:", int(np.isinf(df[engineered]).sum().sum()))

df[engineered].describe().round(2)

Missing values : 0
Infinite values: 0


,theoretical_max_production,machine_worker_ratio,avg_hours_per_worker
count,9495.00,9495.00,9495.00
mean,340.60,0.30,40.93
std,112.20,0.11,3.42
min,125.96,0.08,30.25
25%,256.21,0.22,38.50
50%,322.40,0.28,40.75
75%,409.68,0.36,43.15
max,959.47,0.88,55.90


**How to read this:** both counts should be `0`. In the `describe()` table, check the spread (`std`, `min`, `max`) of each feature - a feature that barely varies, such as an `avg_hours_per_worker` that is almost constant, adds little beyond the columns it was built from.


---
## Step 5 - Remove columns that must not be model inputs
| Column | Why it is removed |
|---|---|
| `planned_qty` | The model forecasts output from operating conditions alone, without being told the plan (the prediction notebook's input deliberately has no `planned_qty`). |
| `production_gap_qty` | Equals `planned_qty − actual_qty` (verified in the EDA), so it would reveal the target directly. |
| `bottleneck_flag` | Excluded as a precaution: the flag may be assigned after the production outcome is known, which would leak the answer. |
| `machine_capacity_per_day` | Constant (200 in every row), so it carries no information. |
| `week_end_date` | Raw date; adds nothing beyond `week_start_date`. |
| `financial_year` | The most recent year is held out for testing, so a year value would be unseen at test time and at prediction time. |

`week_start_date` is **kept** in the saved file only as the ordering key described in Step 2.


In [7]:
target = 'actual_qty'

drop_cols = [
    'planned_qty', 'production_gap_qty', 'bottleneck_flag',
    'machine_capacity_per_day', 'week_end_date', 'financial_year'
]

final_df = df.drop(columns=drop_cols)

# Readable layout: ordering key first, target last
ordered_cols = (
    ['week_start_date']
    + [col for col in final_df.columns if col not in ('week_start_date', target)]
    + [target]
)
final_df = final_df[ordered_cols]

print("Columns removed:", drop_cols)
print("Columns remaining:", final_df.shape[1])

Columns removed: ['planned_qty', 'production_gap_qty', 'bottleneck_flag', 'machine_capacity_per_day', 'week_end_date', 'financial_year']
Columns remaining: 19


---
## Step 6 - Review the final feature set
The remaining input columns fall into three groups. Notebook 2 applies a different transformation to each group, inside the modelling pipeline.

| Group | Columns | Treatment in Notebook 2 |
|---|---|---|
| **Numeric** | `week_no`, `week_of_month`, `rate`, `no_of_workers`, `working_hours`, `total_duration_per_shoe_mins`, `machine_count`, `theoretical_max_production`, `machine_worker_ratio`, `avg_hours_per_worker` | Standard scaling |
| **Low-cardinality categorical** | `financial_month`, `factory_unit`, `product_family`, `grade`, `order_category`, `size_id` | One-hot encoding |
| **High-cardinality categorical** | `item_name` (370 unique items) | Target encoding |

`size_id` is stored as a number but is really a size label, so it is treated as a category. `item_name` is target-encoded because one-hot encoding 370 items would create hundreds of sparse columns.


In [8]:
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']

feature_cols = [col for col in final_df.columns if col not in ('week_start_date', target)]
numerical_cols = [col for col in feature_cols if col not in categorical_ohe + categorical_target]

print(f"Numeric ({len(numerical_cols)}):", numerical_cols)
print(f"One-hot ({len(categorical_ohe)}):", categorical_ohe)
print(f"Target-encoded ({len(categorical_target)}):", categorical_target)
print("\nTotal input features:", len(feature_cols))

Numeric (10): ['week_no', 'week_of_month', 'rate', 'no_of_workers', 'working_hours', 'total_duration_per_shoe_mins', 'machine_count', 'theoretical_max_production', 'machine_worker_ratio', 'avg_hours_per_worker']
One-hot (6): ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
Target-encoded (1): ['item_name']

Total input features: 17


**Expected result:** 10 numeric + 6 one-hot + 1 target-encoded = **17 input features**.


---
## Step 7 - Quality checks and save


In [9]:
assert final_df.isna().sum().sum() == 0, "Missing values found"
assert np.isfinite(final_df[numerical_cols + [target]].to_numpy()).all(), "Infinite values found"
assert final_df['week_start_date'].is_monotonic_increasing, "Rows are not in chronological order"

final_df.to_csv("final_features.csv", index=False)

print("All checks passed.")
print("Saved final_features.csv with shape:", final_df.shape)

final_df.head()

All checks passed.
Saved final_features.csv with shape: (9495, 19)


,week_start_date,financial_month,week_no,week_of_month,factory_unit,product_family,item_name,size_id,grade,rate,no_of_workers,working_hours,total_duration_per_shoe_mins,machine_count,order_category,theoretical_max_production,machine_worker_ratio,avg_hours_per_worker,actual_qty
0,2022-04-18,Apr,3,3,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,12,A,4363.35,15,647.82,114.50,5,Others,339.468996,0.333333,43.188000,38
1,2022-04-18,Apr,3,3,Factory Unit 01,Gents,RITHU 02 - L - BLUE,12,A,4223.86,20,855.19,141.95,6,Others,361.475167,0.300000,42.759500,41
2,2022-04-18,Apr,3,3,Factory Unit 01,Gents,ROME (VAST) - BROWN,9,A,7851.90,21,866.42,124.01,4,Others,419.201677,0.190476,41.258095,37
3,2022-04-18,Apr,3,3,Factory Unit 01,Gents,SH 389 TRUE MOCCASINO - BEIGE,9,B,3170.54,21,805.51,105.90,6,Others,456.379603,0.285714,38.357619,54
4,2022-04-18,Apr,3,3,Factory Unit 01,Gents,SN 002 - L - BROWN,11,A,5207.12,21,864.35,120.46,4,Others,430.524655,0.190476,41.159524,34


---
## Summary
`final_features.csv` contains, per row: the ordering key `week_start_date`, **17 input features** (14 original + 3 engineered) and the target `actual_qty`. Rows are in chronological order, ready for the 80/20 time-based split in Notebook 2.
